# Tutorial 03b

Thomas Manke  
2026-11-12

The goal is to build a multivariate linear model.

------------------------------------------------------------------------

## Setup

Run the setup cell below to confirm package installations and load the
dataset.

- **Data Inspection:** Confirm the sample size $n$ and the number of
  features $p$.
- **Target Variable:** The response variable is `lpsa` (logarithm of
  prostate-specific antigen, $\text{ng/mL}$), a key biomarker of tumor
  burden recorded in men prior to radical prostatectomy, designated as
  `target` in our DataFrame.
- **Variable Documentation:** Clinical feature descriptions originate
  from Stamey et al. (1989) and Hastie, Tibshirani & Friedman’s *The
  Elements of Statistical Learning*.
- **Categorical Encoding (`svi`):** In clinical records, `svi` indicates
  seminal vesicle invasion ($0 = \text{no}, 1 = \text{yes}$); binary
  indicators can be mapped and one-hot encoded to illustrate standard
  categorical pre-processing.

In [1]:
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import sklearn
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import r2_score

sns.set_theme(context="notebook", style="whitegrid")
FIGSIZE = (7, 4)
RANDOM_STATE = 42

# Load dataset
df = None
for p in ["../../data/prostate.csv", "../data/prostate.csv", "data/prostate.csv"]:
    if os.path.exists(p):
        df = pd.read_csv(p)
        break

if df is None:
    url = "https://web.stanford.edu/~hastie/ElemStatLearn/datasets/prostate.data"
    df = pd.read_csv(url, sep="\t")
    if 'Unnamed: 0' in df.columns:
        df = df.drop(columns=['Unnamed: 0'])

if 'train' in df.columns:
    df = df.drop(columns=['train'])

# General template: One-hot encoding of categorical variables with scikit-learn
df['svi_label'] = df['svi'].map({0: 'no', 1: 'yes'})
display(df[['svi_label']].head(5))

ohe = OneHotEncoder(dtype=int, sparse_output=False)
encoded_cats = pd.DataFrame(ohe.fit_transform(df[['svi_label']]), columns=ohe.get_feature_names_out(['svi_label']))
display(encoded_cats.head(5))

# In linear regression with an intercept, one category is absorbed as reference to avoid collinearity
df['svi'] = encoded_cats['svi_label_yes'].astype(int)
df = df.drop(columns=['svi_label'])

# Designate lpsa as standard target variable
df['target'] = df['lpsa']
df = df.drop(columns=['lpsa'])

print(f"Dataset shape: {df.shape} (n = {df.shape[0]} samples, {df.shape[1]-1} features + target)")
df.head()

Dataset shape: (97, 9) (n = 97 samples, 8 features + target)

------------------------------------------------------------------------

## Task 1: Correlations

- **Correlation Heatmap:** Compute and plot the Pearson correlation
  matrix across all features and the target.
- **Feature Ranking:** Rank features by their absolute correlation with
  the target.

> **Dealing with High Correlations**
>
> In a standard data analysis workflow, pairs of features exhibiting
> high mutual correlation would often be pruned early to avoid
> redundancy. However, to directly observe how collinearity impacts
> parameter uncertainty, retain all features for now—we will
> systematically address this in Task 8.

------------------------------------------------------------------------

## Task 2: Multivariate OLS with Scikit-Learn

Use scikit-learn to

- **Model Fitting:** Select the top predictors from Task 1 and fit a
  linear regression (`model.fit()`)
- **Evaluation:** Calculate the $R^2$ score (`r2_score`)

------------------------------------------------------------------------

## Task 3: The Hat Matrix

Use basic numpy to:

- **Design Matrix:** Calculate the design matrix.
- **Hat Matrix:** Compute
  $\mathbf{H} = \mathbf{X} (\mathbf{X}^T \mathbf{X})^{-1} \mathbf{X}^T$.
- **Equivalence Check:** Verify that
  $\hat{\mathbf{y}} = \mathbf{H}\mathbf{y}$ is identical to
  `model.predict(X)`.
- **Algebraic Properties:** Verify that $\mathbf{H}$ is symmetric,
  idempotent, and $\operatorname{Tr}(\mathbf{H}) = p$.

------------------------------------------------------------------------

## Task 4: Vector Decomposition and Pythagoras

- **Vector Construction:** Construct vectors $\mathbf{y}$,
  $\hat{\mathbf{y}}$, $\bar{y}\mathbf{1}$, and residual
  $\boldsymbol{e}$. Verify that $\bar{y}\mathbf{1} - \hat{\mathbf{y}}$
  is orthogonal to $\boldsymbol{e}$.
- **Pythagoras:** Calculate the sums of squares and verify Pythagorean
  identity: $TSS = ESS + RSS$.
- **Coefficient of Determination:** Compute $R^2$ and verify against
  scikit-learn.

------------------------------------------------------------------------

## Task 5: Residual Analysis

- **Normality Check:** Plot the distribution of residuals. Is it
  centered around zero?
- **Orthogonality Check:** Check for each column of the design matrix
  $\mathbf{X}$ that the scalar product with the residual vector
  $\mathbf{e}$ is zero.
- **Diagnostic Plots:** Plot $\mathbf{e}$ against columns of the design
  matrix, and plot residuals against the predicted $\hat{\mathbf{y}}$.

------------------------------------------------------------------------

## Task 6: Parameter and Prediction Uncertainties

- **Parameter Covariance:** Compute the parameter covariance matrix
  $\operatorname{Cov}(\hat{\boldsymbol{\theta}}) = \hat{\sigma}^2 (\mathbf{X}^T \mathbf{X})^{-1}$
  and extract standard errors $\text{SE}(\hat{\theta}_j)$.
- **Leverage and Intervals:** Compute the *leverage*
  $h(\mathbf{x}_i) = H_{ii}$. Compute intervals using $c = 1.96$ for the
  95% interval:
  - Confidence Interval for the mean:
    $\hat{y}(\mathbf{x}_0) \pm c \cdot \hat{\sigma} \sqrt{h(\mathbf{x}_0)}$
  - Prediction Interval for individual observations:
    $\hat{y}(\mathbf{x}_0) \pm c \cdot \hat{\sigma} \sqrt{1 + h(\mathbf{x}_0)}$
- **Prediction Plots:**
  1.  Plot observed $y$ against predicted $\hat{y}$ for observed data
      with the 95% Prediction Interval.
  2.  Select a variable of choice and evaluate predictions across a
      range of new values while keeping all other variables fixed
      (e.g. at their cohort mean). Plot the predicted responses
      $\hat{y}$ along with confidence and prediction intervals for the
      new observations.

------------------------------------------------------------------------

## Task 7: Model Complexity

- **Stepwise Inclusion:** Sequentially add features $k = 1, \dots, p$ in
  order of the correlation ranking from Task 1.
- **$R^2$ Curve:** Compute in-sample $R^2$ for each step and plot $R^2$
  vs. $k$.
- **Optional Check:** Optionally compute Adjusted $R^2$:
  $$R^2_{\text{adj}} = 1 - (1 - R^2) \frac{n - 1}{n - k - 1}$$

------------------------------------------------------------------------

## Task 8: Dealing with Collinearity

- **Feature Pruning:** Remove one of the variables from the strongest
  correlation identified in Task 1.
- **Model Fitting & Covariance:** Fit both the full model (with all
  features) and the pruned model. Compute the parameter covariance
  matrix from first principles:
  $$\operatorname{Cov}(\hat{\boldsymbol{\theta}}) = \hat{\sigma}^2 (\mathbf{X}^T\mathbf{X})^{-1}$$
- **Evaluation:**
  1.  Compare the $R^2$ scores of both models.
  2.  Compare the parameter estimates and their standard errors.

------------------------------------------------------------------------

> **Conceptual Review Questions**
>
> 1.  **Trace of the Hat Matrix:** Using the cyclic property of the
>     trace
>     ($\operatorname{Tr}(\mathbf{A}\mathbf{B}) = \operatorname{Tr}(\mathbf{B}\mathbf{A})$),
>     prove that $\operatorname{Tr}(\mathbf{H}) = p$, where
>     $\mathbf{H} = \mathbf{X}(\mathbf{X}^T\mathbf{X})^{-1}\mathbf{X}^T$
>     and $\mathbf{X} \in \mathbb{R}^{n \times p}$.
> 2.  **Residual Orthogonality:** Why does the normal equation
>     $\mathbf{X}^T(\mathbf{y} - \mathbf{X}\hat{\boldsymbol{\theta}}) = \mathbf{0}$
>     guarantee that residuals are uncorrelated with every feature in
>     $\mathbf{X}$?